# StyleTTS 2 Demo (SPT-CRo)

In [ ]:
%cd ..

### Randomness

In [ ]:
from Modules.pts import set_random_seed

set_random_seed(3407)

### Imports and packages

In [ ]:
# Load packages
import sys
import os
import time
import yaml
import torch
import torch.nn.functional as F
import torchaudio
import librosa
# from munch import munchify

# from models import load_ASR_models, load_F0_models, StyleTTS2
# from Modules.diffusion.sampler import (ADPM2Sampler, DiffusionSampler, KarrasSchedule)
# from Utils.PLBERT.util import load_plbert
# from utils import recursive_munch
# from text_utils import TextCleaner
from Modules.pts import PTS

from tpp_ttstool import TppTtstool

%matplotlib inline
import IPython.display as ipd

### Functions and definitions

In [4]:
# Set username
USER = os.environ["USER"]

TPP_PATH = f"/storage/plzen4-ntis/home/{USER}/GIT_repos/TPP/src"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Set path to TPP
sys.path.insert(0, TPP_PATH)

# Define bin and data for phonemizer
TTSTOOL_BIN_PATH = "./Utils/tts_tool/tts_tool"
TTSTOOL_DATA_PATH = "./Utils/tts_tool/data/frontend_ph-redu.json"

# Experiment dir
EXPDIR = "Exps/SPT-CRo.s180n80_spkenc_concat128"

In [ ]:
# Set up TPP
tpp = TppTtstool("cs-cz", tts_tool_bin=TTSTOOL_BIN_PATH, tts_tool_data=TTSTOOL_DATA_PATH)

# Set up phoneme-to-speech
pts = PTS(
    f"{EXPDIR}/config2.processed.yml",
    f"{EXPDIR}/model4tts.pth",
    t=0.7,
    alpha=0.0,  # 0.3
    beta=0.5,  # 0.7
    diffusion_steps=10,
    embedding_scale=1.0,
    speech_rate=1.0,
    use_glob_noise=False,
    fix_noise_in_ph_string=False,
)

In [18]:
TEST_SENT = "Tohle je opravdu velký pokus, u kterého vůbec netuším, jak dopadne. Snad to alespoň trochu bude připomínat původní hlas."
# TEST_SENT = "tohle je velkI pokus, kterI vUbec nevIm, jak dopadne."

In [11]:
REF_WAVS = {
    "Gott": f"Data/SPT-CRo.cs.n100/Gott-Karel/wavs/Gott-Karel_LY111021-10_1396.92_1410.06.wav",
    "Drábová": f"Data/SPT-CRo.cs.n100/Drábová-Dana/wavs/Drábová-Dana_LS120719-18_2768.82_2785.17.wav",
}
REF_EMBS = {
    "Gott": f"Data/SPT-CRo.cs.n100/Gott-Karel/hasp/Gott-Karel_LY111021-10_1396.92_1410.06.pt",
    "Drábová": f"Data/SPT-CRo.cs.n100/Drábová-Dana/hasp/Drábová-Dana_LS120719-18_2768.82_2785.17.pt",
}

In [29]:
# Prepare phonemizer
tpp.ssml_parse(TEST_SENT)

# Iterate over sentences
for idx, ps in enumerate(tpp.to_sentences_phon()):
    if not ps.strip():  # skip empty phonetic string
        continue
    print(f"{idx}: {ps}")

    # Iterate over speakers
    for k, w in REF_WAVS.items():
        wavs = pts([ps], ref_s=w, spk_emb=REF_EMBS[k])
        print(f"{idx}: {k}")
        display(ipd.Audio(wavs[0], rate=24000))

0: tohle je opravdu velkI pokus, u kterEho vUbec netuSIm, jag dopadne.
0: Gott


0: Drábová


1: snat to alespoJ troxu bude pRipomInat pUvodJI hlas.
1: Gott


1: Drábová
